# Finetuning Data Preparation
Prepare training data for embedding model finetuning on the CLEF-IP 2011 patent retrieval task.

**Pipeline**:
1. Load and inspect train/test topic splits
2. Check for data leakage and language distribution
3. Verify candidate coverage in the FAISS index
4. Compute baseline retrieval metrics
5. Label relevance, fill missing documents, and mine hard negatives
6. Export triplet and contrastive training sets

In [ ]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import sqlmodel as sqlm
from datasets import Dataset
from pyrootutils import setup_root
from tqdm import tqdm

from patent_retrieval import dataset, encoder, utils

In [ ]:
root = setup_root(".")
logger = utils.get_logger(__name__)

## Load Train/Test Topics
Load CLEF-IP 2011 PAC training and test qrels.

In [ ]:
clef_ip = Path(os.environ["CLEF_IP_LOCATION"])
test_topics_path = clef_ip / "02_topics" / "test-pac" / "relass_clef-ip-2011-PAC.txt"
train_topics_path = clef_ip / "02_topics" / "training-pac" / "clef-ip-2011_PACTraining" / "qrels.txt"

In [4]:
train_topics = utils.load_topics_df(train_topics_path)
train_topics

,topic,candidate,score
0,EP-1222910-A2,EP-0172513,1
1,EP-1222910-A2,EP-0290133,1
2,EP-1222910-A2,EP-0321872,1
3,EP-1222910-A2,EP-0443269,1
4,EP-1222910-A2,EP-0516711,1
...,...,...,...
1763,EP-1932653-A1,WO-1999021697,1
1764,EP-1932653-A1,WO-2000056539,1
1765,EP-1935664-A1,EP-0096071,2
1766,EP-1935664-A1,EP-0372837,1


In [5]:
test_topics = utils.load_topics_df(test_topics_path)
test_topics

,topic,candidate,score
0,EP-1221372-A2,EP-0174751,1
1,EP-1221372-A2,EP-0229673,1
2,EP-1221372-A2,EP-0271257,1
3,EP-1221372-A2,EP-0433280,1
4,EP-1221372-A2,EP-0445916,1
...,...,...,...
28595,EP-1936237-A1,EP-0320621,1
28596,EP-1936237-A1,WO-1981001440,2
28597,EP-1936750-A1,EP-0948090,2
28598,EP-1936750-A1,EP-0948092,2


## Train/Test Overlap Analysis
Verify there is no data leakage between training and test splits by checking shared topics, candidates, and exact pairs.

In [6]:
# check overlaps between train_topics and test_topics

topics_train = set(train_topics["topic"])
topics_test = set(test_topics["topic"])
common_topics = topics_train & topics_test

candidates_train = set(train_topics["candidate"])
candidates_test = set(test_topics["candidate"])
common_candidates = candidates_train & candidates_test

pairs_train = set(zip(train_topics["topic"], train_topics["candidate"]))
pairs_test = set(zip(test_topics["topic"], test_topics["candidate"]))
common_pairs = pairs_train & pairs_test


print(f"train topics: {len(topics_train)}, test topics: {len(topics_test)}, common topics: {len(common_topics)}")
print("sample common topics:",list(common_topics)[:10])

print(f"\ntrain candidates: {len(candidates_train)}, test candidates: {len(candidates_test)}, common candidates: {len(common_candidates)}")
print("sample common candidates:",list(common_candidates)[:10])

print(f"\ncommon (topic, candidate) pairs: {len(common_pairs)}")
print("sample common pairs:",list(common_pairs)[:10])

print(
    "Overlaps found - topics: %d, candidates: %d, exact pairs: %d" % (
        len(common_topics),
        len(common_candidates),
        len(common_pairs),
    )
)

train topics: 300, test topics: 3973, common topics: 0
sample common topics: []

train candidates: 1761, test candidates: 26607, common candidates: 212
sample common candidates: ['EP-0107159', 'EP-0924383', 'WO-2001071206', 'EP-1322468', 'EP-0969862', 'EP-1034776', 'WO-1998011777', 'EP-1345570', 'EP-0407870', 'EP-0970390']

common (topic, candidate) pairs: 0
sample common pairs: []
Overlaps found - topics: 0, candidates: 212, exact pairs: 0


## Topic Language Distribution
Profile the language of training topics to understand multilingual coverage.

In [7]:
topic_langs = defaultdict(set)
for topic in tqdm(train_topics["topic"].unique()):
    
    topic_file = next(
            train_topics_path.parent.glob(f"files/{topic}.xml")
    )

    topic_patent = dataset.parse_patent([topic_file])[0]
    topic_langs[topic_patent.language].add(topic_patent.number)


100%|██████████| 300/300 [00:05<00:00, 55.89it/s]


In [ ]:
topic_lang_counter = Counter({lang: len(ids) for lang, ids in topic_langs.items()})
topic_lang_counter

Counter({'DE': 100, 'FR': 100, 'EN': 100})

## Encoder Index Coverage
Load the FAISS index and check how many ground-truth candidates are actually retrievable.

In [ ]:
INDEX_DIR = "/home/alm3rng/scratch/clef_ip_2011/qwen3_emb_4b_v3_title-abstract-claims"

patent_encoder = encoder.get_encoder(
    type="dense", backend="openai", store_type="faiss",
    model_name="Qwen/Qwen3-Embedding-4B", index_dir=INDEX_DIR,
)

[04/02/26 14:05:58] INFO     Encoder backend: openai                                            dense_encoder.py:66

max tokens: 40960


In [ ]:
patent_encoder.load_index(path=INDEX_DIR, store_type="faiss")

[04/02/26 14:06:41] INFO     Loaded FAISS index from:                                          dense_encoder.py:209
                             /home/alm3rng/scratch/clef_ip_2011/qwen3_emb_4b_v3_title-abstract                     
                             -claims                                                                               

In [11]:
index_ids = patent_encoder.get_indices()


In [12]:
# check which candidates from the training dataframe exist in the encoder index
index_id_set = set(index_ids)
train_candidates = set(train_topics["candidate"].unique())

in_index = train_candidates & index_id_set
not_in_index = train_candidates - index_id_set

print(f"total train candidates: {len(train_candidates)}")
print(f"present in index: {len(in_index)}")
print(f"missing from index: {len(not_in_index)}")



total train candidates: 1761
present in index: 1438
missing from index: 323


In [ ]:
train_indexed = train_topics[train_topics["candidate"].isin(index_id_set)]
print(f"topics with indexed candidates: {train_indexed['topic'].nunique()}")

## Baseline Retrieval Metrics
Load retrieval scores from the dense retriever and compute MAP/recall at top-k=300.

In [16]:
scores_df = pd.read_csv("results.csv", sep=",", names=["topic", "candidate", "score","tp"],skiprows=1)
scores_df

,topic,candidate,score,tp
0,EP-1222910-A2,EP-1022012,0.726562,0
1,EP-1222910-A2,EP-0813856,0.711452,0
2,EP-1222910-A2,EP-0486775,0.709973,0
3,EP-1222910-A2,WO-2000078271,0.708247,0
4,EP-1222910-A2,EP-1307171,0.708086,0
...,...,...,...,...
149995,EP-1935664-A1,EP-0695979,0.600026,0
149996,EP-1935664-A1,EP-1194301,0.599917,0
149997,EP-1935664-A1,WO-1994011203,0.599914,0
149998,EP-1935664-A1,EP-0664639,0.599880,0


In [ ]:
metrics = utils.calculate_metrics(scores_df.iloc[:,:3], topk=300,test_topics_path=train_topics_path)

metrics_json = json.dumps(metrics, indent=4)
print(metrics_json)

{'accuracy@10': 0.08433333333333333,
 'precision@10': 0.08433333333333333,
 'recall@10': 0.16831275849875146,
 'f1@10': 0.10591312100316229,
 'nDCG@10': 0.3147561011226063,
 'accuracy@20': 0.05433333333333334,
 'precision@20': 0.05433333333333334,
 'recall@20': 0.21094144538377152,
 'f1@20': 0.08194188558164225,
 'nDCG@20': 0.33300603890601793,
 'accuracy@50': 0.03206666666666667,
 'precision@50': 0.03206666666666667,
 'recall@50': 0.30374643130811163,
 'f1@50': 0.05591032264181981,
 'nDCG@50': 0.354153827329928,
 'accuracy@100': 0.020466666666666668,
 'precision@100': 0.020466666666666668,
 'recall@100': 0.3824502513558349,
 'f1@100': 0.03793919561861676,
 'nDCG@100': 0.3577354494694216,
 'accuracy@200': 0.012316666666666667,
 'precision@200': 0.012316666666666667,
 'recall@200': 0.4546893274004852,
 'f1@200': 0.02363108913151234,
 'nDCG@200': 0.3610692499398796,
 'accuracy@300': 0.009022222222222223,
 'precision@300': 0.009022222222222223,
 'recall@300': 0.4950208811310396,
 'f1@300'

## Label Relevance & Fill Missing Documents
Add true-positive labels to retrieval results and backfill any relevant candidates that were not retrieved (score=0).

In [ ]:
true_dict = utils.load_true_docs(train_topics_path)
scores_df["tp"] = scores_df.apply(
    lambda row: 1 if row["candidate"] in true_dict.get(row["topic"], set()) else 0,
    axis=1,
)
scores_df

,topic,candidate,score,tp
0,EP-1222910-A2,EP-1022012,0.726562,0
1,EP-1222910-A2,EP-0813856,0.711452,0
2,EP-1222910-A2,EP-0486775,0.709973,0
3,EP-1222910-A2,WO-2000078271,0.708247,0
4,EP-1222910-A2,EP-1307171,0.708086,0
...,...,...,...,...
149995,EP-1935664-A1,EP-0695979,0.600026,0
149996,EP-1935664-A1,EP-1194301,0.599917,0
149997,EP-1935664-A1,WO-1994011203,0.599914,0
149998,EP-1935664-A1,EP-0664639,0.599880,0


In [32]:
# Add any true (relevant) candidates that are missing from scores_df with score 0
missing_rows = []
for topic, rels in true_dict.items():
    existing = set(scores_df.loc[scores_df["topic"] == topic, "candidate"])
    for cand in set(rels) - existing:
        missing_rows.append({"topic": topic, "candidate": cand, "score": 0.0, "tp": 1})

if missing_rows:
    scores_df = pd.concat([scores_df, pd.DataFrame(missing_rows)], ignore_index=True)
    # optional: keep a stable ordering (by topic then score desc)
    scores_df["score"] = scores_df["score"].astype(float)
    scores_df = scores_df.sort_values(["topic", "score"], ascending=[True, False]).reset_index(drop=True)
    #scores_df.to_csv("results_filled.csv", index=False)
    print(f"Added {len(missing_rows)} missing rows and saved to results_filled.csv")
else:
    print("No missing candidates to add")

No missing candidates to add


## Sample Positives & Hard Negatives
Per topic:
- **Positives**: up to 2 retrieved relevant docs + 1 missed relevant doc (score=0)
- **Hard negatives**: up to 9 non-relevant docs from ranks 30–100

In [33]:
scores_df = pd.read_csv("results_filled.csv")

In [37]:
finetuning_data = []

for topic in scores_df["topic"].unique():
    topic_df = scores_df[scores_df["topic"] == topic].sort_values("score", ascending=False)
    
    # Get positive candidates (tp=1)
    tp_nonzero = topic_df[(topic_df["tp"] == 1) & (topic_df["score"] != 0)]
    tp_zero = topic_df[(topic_df["tp"] == 1) & (topic_df["score"] == 0)]
    
    # Sample 2 TP with score != 0 and 1 with score == 0
    pos_nonzero = tp_nonzero.sample(n=min(2, len(tp_nonzero)), random_state=42) if len(tp_nonzero) > 0 else pd.DataFrame()
    pos_zero = tp_zero.sample(n=min(1, len(tp_zero)), random_state=42) if len(tp_zero) > 0 else pd.DataFrame()
    
    positives = pd.concat([pos_nonzero, pos_zero])
    
    # Get hard negatives from top 30-100 ranked candidates (tp=0)
    topic_df_ranked = topic_df.reset_index(drop=True)
    hard_neg_pool = topic_df_ranked.iloc[30:100]
    hard_neg_pool = hard_neg_pool[hard_neg_pool["tp"] == 0]
    
    hard_negatives = hard_neg_pool.sample(n=min(9, len(hard_neg_pool)), random_state=42) if len(hard_neg_pool) > 0 else pd.DataFrame()
    
    # Combine for this topic
    topic_samples = pd.concat([positives, hard_negatives])
    topic_samples["sample_type"] = ["positive"] * len(positives) + ["hard_negative"] * len(hard_negatives)
    
    finetuning_data.append(topic_samples)

finetuning_df = pd.concat(finetuning_data, ignore_index=True)
finetuning_df

,topic,candidate,score,tp,sample_type
0,EP-1222910-A2,EP-0172513,0.633459,1,positive
1,EP-1222910-A2,EP-0973482,0.597897,1,positive
2,EP-1222910-A2,EP-0516751,0.000000,1,positive
3,EP-1222910-A2,EP-1343453,0.659829,0,hard_negative
4,EP-1222910-A2,EP-0321841,0.670895,0,hard_negative
...,...,...,...,...,...
3443,EP-1935664-A1,EP-1176566,0.655594,0,hard_negative
3444,EP-1935664-A1,EP-0297014,0.668913,0,hard_negative
3445,EP-1935664-A1,EP-0773507,0.673503,0,hard_negative
3446,EP-1935664-A1,WO-1998029857,0.660430,0,hard_negative


## Structure Triplet Data
Distribute hard negatives across positives to form `{anchor, positive, negatives}` triplets.

In [12]:
# Structure finetuning_df as list of dicts: anchor=topic, positive=one positive candidate, negatives=max 3 hard negatives
json_data = []
for topic, group in finetuning_df.groupby("topic"):
    positives = group[group["sample_type"] == "positive"]["candidate"].tolist()
    negatives = group[group["sample_type"] == "hard_negative"]["candidate"].tolist()
    # keep at most 2 negatives (deterministic: take first 3)
    # distribute negatives across positives without duplication (deterministic)
    neg_pool = negatives[:]  # copy order-preserving list
    # limit pool to at most 2 negatives per positive
    neg_pool = neg_pool[: 2 * max(1, len(positives))]
    for pos in positives:
        take = min(2, len(neg_pool))
        selected = neg_pool[:take]
        json_data.append({"anchor": topic, "positive": pos, "negatives": selected})
        # remove used negatives so next positive gets different ones
        del neg_pool[:take]

In [15]:
json_data

[{'anchor': 'EP-1222910-A2',
  'positive': 'EP-0172513',
  'negatives': ['EP-1343453', 'EP-0321841']},
 {'anchor': 'EP-1222910-A2',
  'positive': 'EP-0973482',
  'negatives': ['EP-0499180', 'EP-0164607']},
 {'anchor': 'EP-1222910-A2',
  'positive': 'WO-1992021632',
  'negatives': ['EP-0413174', 'EP-0366977']},
 {'anchor': 'EP-1224915-A1',
  'positive': 'WO-1998029046',
  'negatives': ['EP-0779796', 'EP-1075224']},
 {'anchor': 'EP-1224915-A1',
  'positive': 'EP-0955930',
  'negatives': ['EP-0592266', 'WO-1999055246']},
 {'anchor': 'EP-1224915-A1',
  'positive': 'EP-0707829',
  'negatives': ['EP-1030617', 'EP-0722298']},
 {'anchor': 'EP-1229507-A2',
  'positive': 'WO-1998019259',
  'negatives': ['EP-1265166', 'EP-1120728']},
 {'anchor': 'EP-1229507-A2',
  'positive': 'WO-2002007021',
  'negatives': ['WO-2002033951', 'WO-2000067210']},
 {'anchor': 'EP-1233422-A1',
  'positive': 'EP-1119004',
  'negatives': ['EP-0858660', 'EP-0537082']},
 {'anchor': 'EP-1233422-A1',
  'positive': 'WO-19900

## Hard Negative Language Analysis
Check the language distribution of sampled hard negatives to detect potential language bias.

In [ ]:
engine = sqlm.create_engine(f"sqlite:///{Path(os.environ['CLEF_IP_LOCATION']) / 'patents_v3.db'}")

In [ ]:
hard_neg_ids = finetuning_df[finetuning_df["sample_type"] == "hard_negative"]["candidate"].unique()

with sqlm.Session(engine) as session:
    hard_neg_langs = session.exec(
        sqlm.select(dataset.Patent.language).where(dataset.Patent.number.in_(hard_neg_ids))
    ).all()

In [ ]:
hard_neg_lang_counter = Counter(hard_neg_langs)
hard_neg_lang_counter

Counter({'EN': 1090, 'DE': 836, 'FR': 753})

## Export Training Sets

### Multiple Negative Ranking Loss
Save triplets as `{anchor, positive, negatives}` for MNRL training.

In [ ]:
with open("finetuning_data.json", "w") as f:
    json.dump(json_data, f, indent=4)

### Standard Contrastive Learning
Convert triplets into `{sentence1, sentence2, label}` pairs for contrastive loss.

In [18]:

# convert json_data (anchor, positive, negatives) into pairs: {sentence1, sentence2, label}
paired = []
for item in json_data:
    anchor = item["anchor"]
    positive = item["positive"]
    # positive pair
    paired.append({"sentence1": anchor, "sentence2": positive, "label": 1})
    # negative pairs
    for neg in item.get("negatives", []):
        paired.append({"sentence1": anchor, "sentence2": neg, "label": 0})

cl_data = json.dumps(paired, indent=4)
with open("cl_data.json", "w") as f:
    f.write(cl_data)


### HuggingFace Dataset

In [ ]:
hf_dataset = Dataset.from_pandas(finetuning_df[["topic", "candidate", "tp"]])
hf_dataset

Dataset({
    features: ['topic', 'candidate', 'tp'],
    num_rows: 3373
})